In [0]:
%pip install yfinance

In [0]:
#Databricks notebook source
# ============================================================================
# STOCK MARKET ANALYTICS: Data Ingestion
# ============================================================================
# Purpose: Ingest real stock market data from Yahoo Finance
# Data Source: yfinance (https://finance.yahoo.com via API)
# Author: RSangDev
# Date: 2026-06-16
# ============================================================================
 
# MAGIC %md
# ## Stock Market Analytics - Data Ingestion
# Ingesting REAL stock price data from Yahoo Finance API
 

In [0]:

import yfinance as yf
from pyspark.sql.window import Window
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import time
 
catalog = "workspace"
schema = "stock_analytics"
 
print("=" * 70)
print("STOCK MARKET PIPELINE - DATA INGESTION")
print("=" * 70)

In [0]:

# Cell 1: Setup
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
print(f"✅ Schema created: {catalog}.{schema}")
 
# Stocks to track (Popular, diverse sectors)
STOCKS = [
    "AAPL",      # Apple (Tech)
    "MSFT",      # Microsoft (Tech)
    "GOOGL",     # Google (Tech)
    "AMZN",      # Amazon (Retail/Cloud)
    "TSLA",      # Tesla (Auto/Energy)
    "META",      # Meta (Social)
    "NVDA",      # Nvidia (semiconductors)
    "JPM",       # JP Morgan (Finance)
    "JNJ",       # Johnson & Johnson (Healthcare)
    "PG",        # Procter & Gamble (Consumer)
]
 
print(f"📊 Tracking {len(STOCKS)} stocks: {', '.join(STOCKS)}")

In [0]:

# Cell 2: Fetch Stock Data from Yahoo Finance
print("\n" + "=" * 70)
print("FETCHING STOCK DATA FROM YAHOO FINANCE")
print("=" * 70 + "\n")
 
# Period: Last 5 years of historical data
all_stocks_data = []
start_date = (datetime.now() - timedelta(days=365*5)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")
 
print(f"Date range: {start_date} to {end_date}\n")
 
for i, stock in enumerate(STOCKS, 1):
    print(f"[{i}/{len(STOCKS)}] Fetching {stock}...", end=" ")
    try:
        # Fetch historical data from Yahoo Finance
        df = yf.download(stock, start=start_date, end=end_date, progress=False)
        
        if df is not None and len(df) > 0:
            # Reset index to make Date a column
            df.reset_index(inplace=True)
            
            # Add stock ticker
            df['Ticker'] = stock
            
            # Rename columns (lowercase for consistency)
            df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Ticker']
            
            # Add metadata
            df['data_fetched_at'] = datetime.now().isoformat()
            df['source'] = 'yahoo_finance'
            
            all_stocks_data.append(df)
            print(f"✅ ({len(df)} records)")
        else:
            print("❌ No data")
            
    except Exception as e:
        print(f"❌ Error: {str(e)}")
    
    time.sleep(0.5)  # Be respectful to API
 
print(f"\n✅ Successfully fetched {len(all_stocks_data)} stocks")
 

In [0]:

# Cell 3: Create Bronze Table - Raw Stock Data
print("\n" + "=" * 70)
print("CREATING BRONZE LAYER")
print("=" * 70 + "\n")
 
# Combine all stocks
combined_df = pd.concat(all_stocks_data, ignore_index=True)
 
# Convert to Spark DataFrame
spark_df = spark.createDataFrame(combined_df)
 
# Add ingestion metadata
spark_df = spark_df.withColumn("ingestion_date", F.current_date())
spark_df = spark_df.withColumn("ingestion_timestamp", F.current_timestamp())
 
# Save to Bronze
bronze_table = f"{catalog}.{schema}.bronze_stock_prices"
spark_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(bronze_table)
 
total_records = spark_df.count()
date_range = spark_df.agg(F.min("Date"), F.max("Date")).collect()[0]
 
print(f"✅ Created: {bronze_table}")
print(f"   Records: {total_records:,}")
print(f"   Date range: {date_range[0]} to {date_range[1]}")
print(f"   Stocks: {', '.join(STOCKS)}")
 

In [0]:

# Cell 4: Data Quality Checks - Bronze Layer
print("\n" + "=" * 70)
print("BRONZE LAYER QUALITY CHECKS")
print("=" * 70 + "\n")
 
bronze_df = spark.table(bronze_table)
 
# Check 1: NULL values
null_count = bronze_df.filter(
    F.col("Close").isNull() | 
    F.col("Volume").isNull() | 
    F.col("Date").isNull()
).count()
 
print(f"✅ NULL check: {null_count} issues")
 
# Check 2: Negative or zero prices (impossible)
negative_prices = bronze_df.filter(
    (F.col("Close") <= 0) | 
    (F.col("High") <= 0) | 
    (F.col("Low") <= 0) |
    (F.col("Open") <= 0)
).count()
 
print(f"✅ Negative prices: {negative_prices} issues")
 
# Check 3: Logical consistency (High >= Low, etc)
logic_issues = bronze_df.filter(
    (F.col("High") < F.col("Low")) |
    (F.col("High") < F.col("Close")) |
    (F.col("Low") > F.col("Close"))
).count()
 
print(f"✅ Logic issues (High<Low): {logic_issues} issues")
 
# Check 4: Volume validation (should be > 0)
zero_volume = bronze_df.filter(F.col("Volume") <= 0).count()
 
print(f"✅ Zero volume days: {zero_volume} issues")
 
# Check 5: Duplicates (same ticker + date)
duplicates = bronze_df.groupBy("Ticker", "Date").count().filter(F.col("count") > 1).count()
 
print(f"✅ Duplicate ticker-date combos: {duplicates} issues")
 
print(f"\n✅ Bronze layer quality: {'PASS ✅' if (null_count + negative_prices + logic_issues + zero_volume + duplicates) == 0 else 'REVIEW ⚠️'}")
 

In [0]:

# Cell 5: Summary Statistics
print("\n" + "=" * 70)
print("DATA SUMMARY")
print("=" * 70 + "\n")
 
summary_stats = bronze_df.groupBy("Ticker").agg(
    F.count("Date").alias("trading_days"),
    F.min("Date").alias("earliest_date"),
    F.max("Date").alias("latest_date"),
    F.avg("Close").alias("avg_close"),
    F.max("Close").alias("highest_close"),
    F.min("Close").alias("lowest_close"),
    F.avg("Volume").alias("avg_volume")
).orderBy(F.desc("latest_date"))
 
print("Stock Summary Statistics:")
display(summary_stats)

In [0]:

# Cell 6: Current Price Snapshot
print("\n" + "=" * 70)
print("LATEST PRICES")
print("=" * 70 + "\n")
 
latest_prices = bronze_df.withColumn(
    "rank", 
    F.row_number().over(
        Window.partitionBy("Ticker").orderBy(F.desc("Date"))
    )
).filter(F.col("rank") == 1).select(
    "Ticker", "Date", "Open", "High", "Low", "Close", "Volume"
).orderBy("Ticker")
 
print("Latest trading data (most recent date):")
display(latest_prices)

In [0]:

# Cell 7: Save Stock Metadata
print("\n✅ INGESTION COMPLETE")
print(f"\nDataset details:")
print(f"  • Total records: {total_records:,}")
print(f"  • Stocks: {len(STOCKS)}")
print(f"  • Date range: 5 years of history")
print(f"  • Latest update: Today")
 
print("\nNEXT STEP: Run 02_technical_indicators.py")